In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"

%load_ext autoreload
%autoreload 2

In [2]:
import ipdb
from Data import *
from address import *
from modelGen import *
from utils import cf_eval, cleanup_gpu
from models import CausalCondLatentCF, latentCFpp, PrototypeLatentCF
from models import CondLatentCF # Note: get the vanilla CondLatentCF
import numpy as np
import random 
from tensorflow import random as tf_random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product
import argparse
import os
import pandas as pd


In [3]:
SEED = 23

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf_random.set_seed(SEED)

CF_METHODS = {  
    "LatentCFpp": latentCFpp,
    "Prototype": PrototypeLatentCF,
    "CausalCACTUS": CausalCondLatentCF,
    "CACTUS": CondLatentCF
}

In [4]:
# Experimentos que a realizar
EXP = [
    {
        "name": "LatentCF++",
        "data": "HELOC",
        "classifier": "./exp/HELOC_class/config.json",
        "AE": "./exp/HELOC_AE/config.json",
        "CFmethod": "LatentCFpp",
        "context": [
            "MOpenAbove15years", 
            "MInFileAbove6years"
        ],
        "epochs": 100,
        "tol": 0.001,
        "target_prob": 0.5,
        "learning_rate": 0.1
    },
    {
        "name": "Prototype C",
        "data": "HELOC",
        "classifier": "./exp/HELOC_class/config.json",
        "AE": "./exp/HELOC_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "MOpenAbove15years", 
            "MInFileAbove6years"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 0.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "Prototype D",
        "data": "HELOC",
        "classifier": "./exp/HELOC_class/config.json",
        "AE": "./exp/HELOC_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "MOpenAbove15years", 
            "MInFileAbove6years"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 100.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "CACTUS",
        "data": "HELOC",
        "classifier": "./exp/HELOC_class/config.json",
        "AE": "./exp/HELOC_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "MOpenAbove15years", 
            "MInFileAbove6years"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        # "dynamicAlpha": True
        "dynamicAlpha": False
    },
    {
        "name": "CausalCACTUS",
        "data": "HELOC",
        "classifier": "./exp/HELOC_class/config.json",
        "AE": "./exp/HELOC_CACTUS/config.json",
        "CFmethod": "CausalCACTUS",
        "context": [
            "MOpenAbove15years", 
            "MInFileAbove6years"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": True
        # "dynamicAlpha": False
    }
]


In [5]:
import re
from collections import defaultdict
# TODO: make a function to convert the causal paths for all the test samples
def build_causal_index_map(input_features, graph_str):
    """
    Returns:
        child_to_parents: dict {child_idx: [parent_idx, ...]}
        parent_to_children: dict {parent_idx: [child_idx, ...]}
        feat2idx: dict {feature_name: index}
    """

    # Feature name to index
    feat2idx = {f: i for i, f in enumerate(input_features)}

    # Extract edges using regex
    edges = re.findall(r'(\w+)\s*->\s*(\w+)', graph_str)

    child_to_parents = defaultdict(list)
    parent_to_children = defaultdict(list)

    for parent, child in edges:

        if parent not in feat2idx or child not in feat2idx:
            raise ValueError(f"Feature in graph not found in dataset: {parent} or {child}")

        p_idx = feat2idx[parent]
        c_idx = feat2idx[child]

        child_to_parents[c_idx].append(p_idx)
        parent_to_children[p_idx].append(c_idx)

    return dict(child_to_parents), dict(parent_to_children), feat2idx


# credit_rex5_constraints_min.dot
graph_str1 = """
strict digraph {
NumRevolvingTradesWBalance;
MaxDelqEver;
NumSatisfactoryTrades;
MSinceMostRecentInqexcl7days;
NumTotalTrades;
NumTrades90Ever2DerogPubRec;
NetFractionRevolvingBurden;
PercentInstallTrades;
MaxDelq2PublicRecLast12M;
MSinceMostRecentDelq;
PercentTradesNeverDelq;
MSinceOldestTradeOpen;
NumTradesOpeninLast12M;
NetFractionInstallBurden;
AverageMInFile;
PercentTradesWBalance;
NumTrades60Ever2DerogPubRec;
NumInqLast6M;
NumBank2NatlTradesWHighUtilization;
NumInstallTradesWBalance;
NumSatisfactoryTrades -> MaxDelq2PublicRecLast12M;
MSinceMostRecentInqexcl7days -> NumTradesOpeninLast12M;
MSinceMostRecentInqexcl7days -> NumBank2NatlTradesWHighUtilization;
MSinceMostRecentInqexcl7days -> NumRevolvingTradesWBalance;
MSinceMostRecentInqexcl7days -> PercentTradesWBalance;
NumTotalTrades -> PercentTradesWBalance;
NumTrades90Ever2DerogPubRec -> PercentTradesNeverDelq;
NumTrades90Ever2DerogPubRec -> MaxDelq2PublicRecLast12M;
NetFractionRevolvingBurden -> PercentInstallTrades;
NetFractionRevolvingBurden -> NumRevolvingTradesWBalance;
NetFractionRevolvingBurden -> PercentTradesWBalance;
PercentTradesNeverDelq -> MSinceMostRecentDelq;
PercentTradesNeverDelq -> MaxDelq2PublicRecLast12M;
PercentTradesNeverDelq -> MaxDelqEver;
NetFractionInstallBurden -> NetFractionRevolvingBurden;
NetFractionInstallBurden -> PercentTradesWBalance;
PercentTradesWBalance -> NumTradesOpeninLast12M;
NumTrades60Ever2DerogPubRec -> MaxDelqEver;
NumInqLast6M -> MaxDelqEver;
NumInqLast6M -> NumTradesOpeninLast12M;
NumBank2NatlTradesWHighUtilization -> NetFractionRevolvingBurden;
NumInstallTradesWBalance -> NumTotalTrades;
NumInstallTradesWBalance -> PercentInstallTrades;
NumInstallTradesWBalance -> NumInqLast6M;
NumInstallTradesWBalance -> NetFractionRevolvingBurden;
NumInstallTradesWBalance -> NumRevolvingTradesWBalance;
}
"""

# adult_rex6_constraints_mod.dot
graph_str2 = """
strict digraph {
NumRevolvingTradesWBalance;
MaxDelqEver;
NumSatisfactoryTrades;
MSinceMostRecentInqexcl7days;
NumTotalTrades;
NumTrades90Ever2DerogPubRec;
NetFractionRevolvingBurden;
PercentInstallTrades;
MaxDelq2PublicRecLast12M;
MSinceMostRecentDelq;
PercentTradesNeverDelq;
MSinceOldestTradeOpen;
NumTradesOpeninLast12M;
NetFractionInstallBurden;
AverageMInFile;
PercentTradesWBalance;
NumTrades60Ever2DerogPubRec;
NumInqLast6M;
NumBank2NatlTradesWHighUtilization;
NumInstallTradesWBalance;
NumRevolvingTradesWBalance -> MaxDelq2PublicRecLast12M;
NumSatisfactoryTrades -> MaxDelqEver;
MSinceMostRecentInqexcl7days -> NumBank2NatlTradesWHighUtilization;
MSinceMostRecentInqexcl7days -> PercentTradesWBalance;
NetFractionRevolvingBurden -> NumInqLast6M;
PercentInstallTrades -> MaxDelqEver;
MSinceMostRecentDelq -> MSinceMostRecentInqexcl7days;
MSinceMostRecentDelq -> NumInqLast6M;
PercentTradesNeverDelq -> PercentTradesWBalance;
NetFractionInstallBurden -> MSinceMostRecentInqexcl7days;
NetFractionInstallBurden -> NetFractionRevolvingBurden;
PercentTradesWBalance -> NumTradesOpeninLast12M;
PercentTradesWBalance -> MaxDelqEver;
PercentTradesWBalance -> NumInqLast6M;
NumBank2NatlTradesWHighUtilization -> MaxDelqEver;
}
"""

# # CF generation
# child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
#     data.features_lbls,
#     # graph_str
#     graph_str1
# )
# print("Feature → index:")
# print(feat2idx)
# print("\nChild → Parents (index form):")
# print(child_to_parents_dict)

In [6]:
# data.features_lbls

## Graph 1 (Minimum priors)

In [7]:
N = 50
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [12, 23, 34, 45, 56]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str1
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              

        # # TODO: One-hot encoding function for datasets with catergorical features
        # cat_idx = [2, 3, 4, 5] #    CheckingAccount", "Job", "Housing", "SavingAccounts"
        # num_idx = [0, 1]
        # X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])
        
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict
            # cat_idx,
            # num_idx
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)




********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data
MOpenAbove15years
0    5387
1    5072
Name: count, dtype: int64
MInFileAbove6years
1    5389
0    5070
Name: count, dtype: int64




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Total samples: 10000
Positive class: 5000.0
Context 'MOpenAbove15years=1': 4885
Context 'MInFileAbove6years=1': 5198
----------------------------------------------------------------------------------------------------




Building model


/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_HELOC.py:139: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["MOpenAbove15years"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_HELOC.py:140: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["MInFileAbove6years"]))


Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 18)]              0         
                                                                 
 dense (Dense)               (None, 64)                1216      
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 128)               8320      
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                 

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


restoring weights of model AE: HELOC_AE
Feature → index:
{'NumSatisfactoryTrades': 0, 'NumTrades60Ever2DerogPubRec': 1, 'NumTrades90Ever2DerogPubRec': 2, 'PercentTradesNeverDelq': 3, 'MSinceMostRecentDelq': 4, 'MaxDelq2PublicRecLast12M': 5, 'MaxDelqEver': 6, 'NumTotalTrades': 7, 'NumTradesOpeninLast12M': 8, 'PercentInstallTrades': 9, 'MSinceMostRecentInqexcl7days': 10, 'NumInqLast6M': 11, 'NetFractionRevolvingBurden': 12, 'NetFractionInstallBurden': 13, 'NumRevolvingTradesWBalance': 14, 'NumInstallTradesWBalance': 15, 'NumBank2NatlTradesWHighUtilization': 16, 'PercentTradesWBalance': 17}

Child → Parents (index form):
{5: [0, 2, 3], 8: [10, 17, 11], 16: [10], 14: [10, 12, 15], 17: [10, 7, 12, 13], 3: [2], 9: [12, 15], 4: [3], 6: [3, 1, 11], 12: [13, 16, 15], 7: [15], 11: [15]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(No

...

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 18)]         0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1796        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 18)           1810        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_HELOC.py:139: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["MOpenAbove15years"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_HELOC.py:140: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["MInFileAbove6years"]))


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 18)]              0         
                                                                 
 dense (Dense)               (None, 64)                1216      
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 128)               8320      
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                                             

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 18)]         0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1796        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 18)           1810        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

In [8]:
child_to_parents_dict

{5: [0, 2, 3],
 8: [10, 17, 11],
 16: [10],
 14: [10, 12, 15],
 17: [10, 7, 12, 13],
 3: [2],
 9: [12, 15],
 4: [3],
 6: [3, 1, 11],
 12: [13, 16, 15],
 7: [15],
 11: [15]}

In [9]:
df_metrics

,Model,Dataset,CFMethod,Metric,Value
0,LatentCF++,HELOC,LatentCFpp,proximity,1.608
1,LatentCF++,HELOC,LatentCFpp,n_proximity,0.379
2,LatentCF++,HELOC,LatentCFpp,validity,1.000
3,LatentCF++,HELOC,LatentCFpp,compactness,0.039
4,LatentCF++,HELOC,LatentCFpp,lof_context,1.199
...,...,...,...,...,...
220,CausalCACTUS,HELOC,CausalCACTUS,lof_context,1.229
221,CausalCACTUS,HELOC,CausalCACTUS,causal_validity_hard,0.400
222,CausalCACTUS,HELOC,CausalCACTUS,causal_validity_soft,0.927
223,CausalCACTUS,HELOC,CausalCACTUS,causal_compact_val,0.389


In [10]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                   Validity          LOF  Compactness    Proximity  \
Dataset Model                                                              
HELOC   CACTUS        0.54 ± 0.01  1.22 ± 0.08  0.08 ± 0.03  0.49 ± 0.04   
        CausalCACTUS  0.76 ± 0.04  1.22 ± 0.08  0.27 ± 0.04  0.39 ± 0.03   
        LatentCF++      1.0 ± 0.0  1.32 ± 0.08  0.04 ± 0.01  0.45 ± 0.04   
        Prototype C     1.0 ± 0.0  2.32 ± 0.06  0.05 ± 0.01  1.29 ± 0.03   
        Prototype D     1.0 ± 0.0  1.62 ± 0.04  0.02 ± 0.01  1.08 ± 0.04   

Metric               Causal Validity (Hard) Causal Validity (Soft)  \
Dataset Model                                                        
HELOC   CACTUS                  0.22 ± 0.04             0.9 ± 0.01   
        CausalCACTUS            0.38 ± 0.06            0.93 ± 0.01   
        LatentCF++              0.06 ± 0.03            0.88 ± 0.01   
        Prototype C              0.7 ± 0.09            0.96 ± 0.01   
        Prototype D              0.7 ± 0.07            0.97 ± 0.01   

Metric               Causal Compact Validity inlier_fraction  
Dataset Model                                                 
HELOC   CACTUS                   0.21 ± 0.03     0.95 ± 0.02  
        CausalCACTUS             0.37 ± 0.03     0.96 ± 0.02  
        LatentCF++               0.16 ± 0.01     0.88 ± 0.04  
        Prototype C              0.09 ± 0.01     0.09 ± 0.07  
        Prototype D              0.06 ± 0.01     0.46 ± 0.09

In [11]:
print(data.scaler_inverse_transform(X_test))
print(data.scaler_inverse_transform(X_cf))

[[ 34.   1.   0.  97.  30.   6.   5.  45.   6.  11.   0.   2.  11.  88.
    6.   2.   1.  33.]
 [ 32.   1.   0.  82.   5.   4.   5.  33.   4.  39.   0.   1.  43. 101.
    5.   4.   0.  90.]
 [  6.   0.   0. 100.  -7.   7.   8.  14.   1.  33.   3.   2.  14.  -8.
    1.  -8.   0.  33.]
 [ 28.   4.   4.  97.  56.   6.   4.  31.   8.  29.  -7.   4.  54.  58.
    6.   2.   0.  71.]
 [  4.   2.   2.  88.  23.   6.   4.   8.   4.  88.   2.   1.  -8. 103.
   -8.   4.  -8. 100.]
 [ 16.   0.   0. 100.  -7.   7.   8.  17.   1.  35.   0.   1.   6.  -8.
    7.  -8.   0. 100.]
 [ 14.   0.   0.  93.  -7.   6.   7.  15.   0.  47.  11.   0.  11.  -8.
    1.   2.   0.  75.]
 [ 24.   0.   0.  92.   5.   4.   6.  24.   2.  38.   0.   6.  57.  90.
    9.   3.   2.  86.]
 [ 24.   0.   0. 100.  -7.   7.   8.  27.   2.  26.   0.   2.   2.  76.
    2.   2.   0.  50.]
 [ 26.   0.   0. 100.  -7.   7.   8.  27.   2.  26.   0.   3.  70.  68.
    5.   3.   3.  80.]
 [ 14.   2.   2.  87.  27.   6.   6.  17.   2.  47

In [12]:
# # TODO: One-hot encoding function for datasets with catergorical features
# cat_idx = [2, 3, 4, 5] #    CheckingAccount", "Job", "Housing", "SavingAccounts"
# num_idx = [0, 1]
# X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])


# --- CF evaluation ---
cf_scores, cf_scores_labels = cf_eval(
    data.X_train, # previous name: data.X_train
    context_training, # previous name: context_training,
    X_test, # previous name: x,
    X_cf, # previous name: cf_x, 
    y_original_labels, # previous name: y_,
    context_test, # previous name: context,
    y_cf_labels, # previous name: cf_y_
    data.scaler_inverse_transform(X_test),
    data.scaler_inverse_transform(X_cf),
    child_to_parents_dict,
    # cat_idx,
    # num_idx
)

print(cf_scores, cf_scores_labels)

[0 1 0 1 1 0 0 1 0 1 1 1 1 1 0 1 0 0 1 1 1 1 1 1 0 0 1 0 0 0 1 0 1 0 0 1 1
 1 1 1 1 1 1 1 0 0 0 1 0 1]
[0 1 0 1 1 0 0 1 1 1 0 1 0 1 0 0 0 0 1 1 0 0 1 1 1 0 0 0 0 0 1 0 1 0 0 1 0
 0 1 1 0 0 1 1 0 1 0 1 0 1]
0.74
---- DEBUG Context-LOF score ----
Context: [0, 0]
Subset size: 3114
Unique rows: 2662
After np.unique(): subset size: 2662
cf_samples_ shape: (21, 18)
X_ shape: (2662, 18)
lof_cf min/max: 0.9669658922228529 3.753302927545895
lof_train min/max: 0.9572709929810846 4.949122384816732
mean train LOF: 1.1170444594235096
---- DEBUG Context-LOF score ----
Context: [0, 1]
Subset size: 1021
Unique rows: 1021
After np.unique(): subset size: 1021
cf_samples_ shape: (8, 18)
X_ shape: (1021, 18)
lof_cf min/max: 0.9658655732488753 1.2721821422283197
lof_train min/max: 0.9578591868059378 1.7940915147540137
mean train LOF: 1.108941553229546
---- DEBUG Context-LOF score ----
Context: [1, 0]
Subset size: 766
Unique rows: 766
After np.unique(): subset size: 766
cf_samples_ shape: (4, 18)
X_ shape: 

## Graph 2 (Moderate priors)

In [13]:
N = 50
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [11, 22, 33, 44, 55]

# Number of samples to compute the metrics for CF evaluation
METRICS = []

for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str2
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              

        # # TODO: One-hot encoding function for datasets with catergorical features
        # cat_idx = [2, 3, 4, 5] #    CheckingAccount", "Job", "Housing", "SavingAccounts"
        # num_idx = [0, 1]
        # X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])
        
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict
            # cat_idx,
            # num_idx
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)




********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data
MOpenAbove15years
0    5387
1    5072
Name: count, dtype: int64
MInFileAbove6years
1    5389
0    5070
Name: count, dtype: int64




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Total samples: 10000
Positive class: 5000.0
Context 'MOpenAbove15years=1': 4885
Context 'MInFileAbove6years=1': 5198
----------------------------------------------------------------------------------------------------




Building model
Building model: DNN [class]


/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_HELOC.py:139: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["MOpenAbove15years"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_HELOC.py:140: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["MInFileAbove6years"]))


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 18)]              0         
                                                                 
 dense (Dense)               (None, 64)                1216      
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 128)               8320      
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                                             

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Feature → index:
{'NumSatisfactoryTrades': 0, 'NumTrades60Ever2DerogPubRec': 1, 'NumTrades90Ever2DerogPubRec': 2, 'PercentTradesNeverDelq': 3, 'MSinceMostRecentDelq': 4, 'MaxDelq2PublicRecLast12M': 5, 'MaxDelqEver': 6, 'NumTotalTrades': 7, 'NumTradesOpeninLast12M': 8, 'PercentInstallTrades': 9, 'MSinceMostRecentInqexcl7days': 10, 'NumInqLast6M': 11, 'NetFractionRevolvingBurden': 12, 'NetFractionInstallBurden': 13, 'NumRevolvingTradesWBalance': 14, 'NumInstallTradesWBalance': 15, 'NumBank2NatlTradesWHighUtilization': 16, 'PercentTradesWBalance': 17}

Child → Parents (index form):
{5: [14], 6: [0, 9, 17, 16], 16: [10], 17: [10, 3], 11: [12, 4, 17], 10: [4, 13], 12: [13], 8: [17]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(None, 16)]              0         
                                                                 
 m

...

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 18)]         0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1796        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 18)           1810        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_HELOC.py:139: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["MOpenAbove15years"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_HELOC.py:140: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["MInFileAbove6years"]))


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 18)]              0         
                                                                 
 dense (Dense)               (None, 64)                1216      
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 128)               8320      
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                                             

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 18)]         0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1796        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 18)           1810        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

In [14]:
child_to_parents_dict

{5: [14],
 6: [0, 9, 17, 16],
 16: [10],
 17: [10, 3],
 11: [12, 4, 17],
 10: [4, 13],
 12: [13],
 8: [17]}

In [15]:
df_metrics

,Model,Dataset,CFMethod,Metric,Value
0,LatentCF++,HELOC,LatentCFpp,proximity,1.556
1,LatentCF++,HELOC,LatentCFpp,n_proximity,0.367
2,LatentCF++,HELOC,LatentCFpp,validity,1.000
3,LatentCF++,HELOC,LatentCFpp,compactness,0.048
4,LatentCF++,HELOC,LatentCFpp,lof_context,1.345
...,...,...,...,...,...
220,CausalCACTUS,HELOC,CausalCACTUS,lof_context,1.193
221,CausalCACTUS,HELOC,CausalCACTUS,causal_validity_hard,0.880
222,CausalCACTUS,HELOC,CausalCACTUS,causal_validity_soft,0.985
223,CausalCACTUS,HELOC,CausalCACTUS,causal_compact_val,0.309


In [16]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                   Validity          LOF  Compactness    Proximity  \
Dataset Model                                                              
HELOC   CACTUS        0.62 ± 0.09  1.17 ± 0.07  0.07 ± 0.02   0.5 ± 0.02   
        CausalCACTUS   0.8 ± 0.06  1.18 ± 0.07  0.23 ± 0.06   0.4 ± 0.04   
        LatentCF++      1.0 ± 0.0  1.28 ± 0.06  0.04 ± 0.01  0.44 ± 0.06   
        Prototype C     1.0 ± 0.0  2.37 ± 0.05  0.05 ± 0.01  1.29 ± 0.05   
        Prototype D     1.0 ± 0.0  1.65 ± 0.03  0.03 ± 0.01  1.11 ± 0.03   

Metric               Causal Validity (Hard) Causal Validity (Soft)  \
Dataset Model                                                        
HELOC   CACTUS                  0.81 ± 0.02             0.98 ± 0.0   
        CausalCACTUS            0.84 ± 0.06            0.98 ± 0.01   
        LatentCF++              0.75 ± 0.06            0.97 ± 0.01   
        Prototype C             0.72 ± 0.05            0.96 ± 0.01   
        Prototype D               0.8 ± 0.1            0.97 ± 0.01   

Metric               Causal Compact Validity inlier_fraction  
Dataset Model                                                 
HELOC   CACTUS                   0.21 ± 0.02     0.96 ± 0.02  
        CausalCACTUS             0.34 ± 0.05     0.96 ± 0.02  
        LatentCF++               0.18 ± 0.02     0.88 ± 0.06  
        Prototype C              0.09 ± 0.01     0.12 ± 0.06  
        Prototype D              0.07 ± 0.01     0.42 ± 0.15